In [2]:
import requests
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import gradio as gr

# tokenizer and initialisation of the model
model_id = "microsoft/Phi-4-mini-instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# NewsAPI Key
NEWSAPI_KEY = "6e389e4384a0487ba07861d144b329c8"

def get_latest_disaster_news(query="\"+natural disaster\"", max_articles=3):
    url = (
        f"https://newsapi.org/v2/everything?"
        f"q={query}&"
        f"language=en&"
        f"sortBy=publishedAt&"
        f"pageSize={max_articles}&"
        f"apiKey={NEWSAPI_KEY}"
    )
    response = requests.get(url)
    data = response.json()

    if data.get("status") != "ok" or not data.get("articles"):
        return "No recent news found."

    summaries = []
    for article in data["articles"]:
        title = article.get("title", "")
        desc = article.get("description", "")
        summaries.append(f"- {title}: {desc}")
        
    print(summaries)
    return "\n".join(summaries)

# strict system prompt for natural disaster
system_prompt = (
    "You are an AI assistant specialized strictly in natural disasters. "
    "Answer only questions about natural disasters, safety, preparedness, and recent events. "
    "If the question is unrelated, politely refuse."
)

def chat_with_model(user_input, chat_history=None):
    if chat_history is None:
        chat_history = []

    # 1.First try : answer from the model without research
    messages = [{"role": "system", "content": system_prompt}] + chat_history + [{"role": "user", "content": user_input}]

    chat_input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    chat_input_tokens = tokenizer(chat_input_text, return_tensors="pt").to(model.device)
    input_length = chat_input_tokens["input_ids"].shape[1]
    max_tokens = min(1024, 4096 - input_length)

    outputs = model.generate(
        **chat_input_tokens,
        max_new_tokens=max_tokens,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    generated_tokens = outputs[0][input_length:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    # 2. Verification of the answer
    if len(response) < 20 or "I don't know" in response or "I am not able" in response or "2023" in response or "I'm sorry," in response:
        
        # do research
        news_summary = get_latest_disaster_news()
        context = f"Here are the latest news about natural disasters:\n{news_summary}\n\nQuestion: {user_input}"

        messages = [{"role": "system", "content": system_prompt}] + chat_history + [{"role": "user", "content": context}]

        chat_input_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        chat_input_tokens = tokenizer(chat_input_text, return_tensors="pt").to(model.device)
        input_length = chat_input_tokens["input_ids"].shape[1]
        max_tokens = min(1024, 4096 - input_length)

        outputs = model.generate(
            **chat_input_tokens,
            max_new_tokens=max_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        generated_tokens = outputs[0][input_length:]
        response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    chat_history.append({"role": "user", "content": user_input})
    chat_history.append({"role": "assistant", "content": response})

    return chat_history, chat_history



with gr.Blocks() as demo:
    chatbot = gr.Chatbot(type="messages",height=560)
    msg = gr.Textbox(label="Pose ta question")
    state = gr.State([])

    msg.submit(chat_with_model, inputs=[msg, state], outputs=[chatbot, state])
    msg.submit(lambda: "", None, msg)

demo.launch(share=True)


c:\Users\noxra\KMUTNB_internship\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\noxra\.cache\huggingface\hub\models--microsoft--Phi-4-mini-instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading checkpoint shards: 100%|██████████| 2/2 [00:09<00:00,  4.85s/it]


* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
